In [1]:
import pandas as pd
import numpy as np
import csv
import matplotlib as mpl 
import matplotlib.pyplot as plt

### deseq OUT

In [4]:
# podejcie MW #
#--------------#
# wyniki deseq:
# pvalue jest normalizowane do padj przez "the Holm method" http://www.math.ucla.edu/~anderson/rw1001/library/base/html/p.adjust.html
# w analizie odciecie jest przy padj = 0.001 (mozliwe bylo w DEBROWSER) i taki mam plik wynikowy, podzielony na UP/DOWN
# (w zasadzie dwa pliki), z ktorych na podstawie najwyzszego log2foldchange wzialem po 100 najwyzszch sekwencji,
# polaczylem razem i zblastowalem. Plik ktory jest ponizej to zlepek 100UP i 100DOWN wynikow deseq, tych samych, ktore sie blastowaly

In [2]:
df_deseq_out = pd.read_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/PBS1d_vs_MC1d_DESEQ_100UP_100DOWN_together.csv', sep='\t')
df_deseq_out.head()

,ID,PBS_1d_1,PBS_1d_2,PBS_1d_3,PBS_1d_4,MC_1d_1,MC_1d_2,MC_1d_3,padj,log2FoldChange,pvalue,foldChange,log10padj
0,TRINITY_DN96009_c1_g3_i3,0.0,0.0,0.0,0.000000,1926.746939,838.569902,912.024085,1.819644e-43,12.791799,2.678902e-47,7091.127494,42.740014
1,TRINITY_DN96009_c1_g7_i1,0.0,0.0,0.0,0.000000,1545.529985,939.097763,954.080531,2.081684e-60,12.695019,1.094528e-64,6631.036435,59.681585
2,TRINITY_DN96413_c4_g3_i1,0.0,0.0,0.0,0.000000,1395.729284,899.724351,217.491910,4.846018e-22,12.242750,5.707493e-25,4846.575702,21.314615
3,TRINITY_DN80414_c8_g1_i6,0.0,0.0,0.0,0.000000,373.985197,226.187686,1210.024049,2.914640e-20,11.768755,4.260318e-23,3489.378944,19.535415
4,TRINITY_DN91051_c3_g1_i1,0.0,0.0,0.0,0.905415,1429.821857,965.067460,561.153159,7.112150e-42,11.755678,1.271429e-45,3457.893280,41.147999


# filtrowanie pliku wynikowego blasta

In [3]:
df_blast_results = pd.read_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/blast_salmo_ony.csv', sep='\t', low_memory=False)
df_blast_results.shape

(5205, 5)

### odfiltrowanie zbyt niskiego coverage (pomijamy w tej wersji salmo only)

### filtrowanie mRNA only

In [4]:
# select best hit based on evalue (drop_duplicates)
df_best_hit = df_blast_results.sort_values('evalue', ascending=True).drop_duplicates('query').sort_index()


In [5]:
df_best_hit.shape

(130, 5)

In [6]:
# select what you need based on string

# w tym wypadku mRNA only oznacza pominiecie blastów z ncRNA, mixRNA i innych dziwnych, nie oznacza natomiast
# zabawy z chromosome i filtrowaniem tego dodatkowo, poniewaz blastujac tylko do salmo salar, nie mam (chyba)
# zadnych hitow chromosome. Chromosome (czyli hity bez mRNA) w tym wypadku to takie, ktore w blascie mialy NO HIT.
# czyli zamiast filtrowania po chromosome nalezy wrzucic w te grupe trinity, ktore mialy NO HIT.

df_mRNA_only = df_best_hit[df_best_hit['al_title'].str.contains('mRNA')]
df_mRNA_only.shape


(117, 5)

In [7]:
df_mRNA_only.to_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/PBS1d_vs_MCLR1d_mRNA_only_wo_cov20.csv', index=False)

In [9]:
# łączenie odfiltrowanych mRNA z wynikami deseq

df_mRNA_only.rename(columns={'query' : 'ID'}, inplace=True)
df_mRNA_only.head()

/home/maciek/anaconda3/lib/python3.7/site-packages/pandas/core/frame.py:3781: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  return super(DataFrame, self).rename(**kwargs)


,ID,al_title,al_accession,evalue,coverage
3,TRINITY_DN96009_c1_g7_i1,gi|185134006|ref|NM_001123570.1| Salmo salar i...,NM_001123570,8.331800e-27,20.718232
4,TRINITY_DN91919_c2_g1_i1,gi|929293431|ref|XM_014168489.1| PREDICTED: Sa...,XM_014168489,8.099330e-47,47.592068
6,TRINITY_DN93589_c2_g1_i2,gi|929080070|ref|XM_014194976.1| PREDICTED: Sa...,XM_014194976,8.156840e-42,11.849711
110,TRINITY_DN104041_c7_g1_i1,gi|213514673|ref|NM_001139927.1| Salmo salar G...,NM_001139927,7.335590e-117,100.000000
113,TRINITY_DN90356_c5_g2_i1,gi|929058649|ref|XM_014126149.1| PREDICTED: Sa...,XM_014126149,6.440880e-28,5.921788


In [10]:
result = pd.merge(df_deseq_out, df_mRNA_only, on='ID')
result.sort_values(by='log2FoldChange', inplace=True)

In [11]:
result.to_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/mRNA_PBS_1d_vs_MC2_1d_deseq_w_mRNA_names.csv', index=False)

# TO DO: XM z tego ^^ pozamieniac na XP w unirefie

### filtrowanie hitów innych niz mRNA: ncRNA, miscRNA. ZDECYDOWAC CO Z NIMI

In [17]:
non_mRNA = df_best_hit[~df_best_hit['al_title'].str.contains('mRNA')]


In [18]:

non_mRNA.shape

(13, 5)

In [19]:
result2 = pd.merge(df_deseq_out, non_mRNA, on='ID')
result2.sort_values(by='log2FoldChange', inplace=True)

In [21]:
result2.to_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/non_mRNA_PBS_1d_vs_MC2_1d_deseq_w_mRNA_names.csv', index=False)

### filtrowanie NO HIT
#### chromosome poprzednio == no hit teraz

In [22]:
# wszystkie blastowane trinity (100UP + 100DOWN)
all_trinity_to_blast = pd.read_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/100_UP_100_DOWN_best_fold_change.headers')

In [23]:
all_trinity_to_blast.shape

(200, 1)

In [26]:
# wszystkie odjac te ktore mialy hit w blascie, jakikolwiek hit
no_hit = all_trinity_to_blast[~all_trinity_to_blast['id'].isin(df_best_hit['ID'])]

In [27]:
no_hit.shape

(70, 1)

In [30]:
no_hit.rename(columns={'id' : 'ID'}, inplace=True)

/home/maciek/anaconda3/lib/python3.7/site-packages/pandas/core/frame.py:3781: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  return super(DataFrame, self).rename(**kwargs)


In [31]:
# lista NO HIT czyli dawna chromosomes, ktora idzie do analizy novel lncRNA
result3 = pd.merge(df_deseq_out, no_hit, on='ID')
result3.sort_values(by='log2FoldChange', inplace=True)

In [32]:
result3.to_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/NO_HIT_PBS_1d_vs_MC2_1d_deseq_w_mRNA_names.csv', index=False)